Diabetes Health Indicators Dataset
==========================

Research Question:
Given this Survey Data set can we identify, using feature selection, a subset of survey questions and/or risk factors which predict diabetes. In other words, can we design a more optimal survey which is just as informative about diabetes? 



This is the Diabetes Health Indicators Dataset. I'll let a generous quotation from the Kaggle page tell you about the data:

> The Behavioral Risk Factor Surveillance System (BRFSS) is a health-related telephone survey that is collected annually by the CDC. Each year, the survey collects responses from over 400,000 Americans on health-related risk behaviors, chronic health conditions, and the use of preventative services. It has been conducted every year since 1984. For this project, a csv of the dataset available on Kaggle for the year 2015 was used. This original dataset contains responses from 441,455 individuals and has 330 features. These features are either questions directly asked of participants, or calculated variables based on individual participant responses.

>This dataset contains 3 files:

>    diabetes_012_health_indicators_BRFSS2015.csv is a clean dataset of 253,680 survey responses to the CDC's BRFSS2015. The target variable Diabetes_012 has 3 classes. 0 is for no diabetes or only during pregnancy, 1 is for prediabetes, and 2 is for diabetes. There is class imbalance in this dataset. This dataset has 21 feature variables
    diabetes_ binary_5050split_health_indicators_BRFSS2015.csv is a clean dataset of 70,692 survey responses to the CDC's BRFSS2015. It has an equal 50-50 split of respondents with no diabetes and with either prediabetes or diabetes. The target variable Diabetes_binary has 2 classes. 0 is for no diabetes, and 1 is for prediabetes or diabetes. This dataset has 21 feature variables and is balanced.
    diabetes_binary_health_indicators_BRFSS2015.csv is a clean dataset of 253,680 survey responses to the CDC's BRFSS2015. The target variable Diabetes_binary has 2 classes. 0 is for no diabetes, and 1 is for prediabetes or diabetes. This dataset has 21 feature variables and is not balanced.

The Kaggle Page also makes some useful suggestions for where to start:

> Explore some of the following research questions:

>    Can survey questions from the BRFSS provide accurate predictions of whether an individual has diabetes?
    What risk factors are most predictive of diabetes risk?
    Can we use a subset of the risk factors to accurately predict whether an individual has diabetes?
    Can we create a short form of questions from the BRFSS using feature selection to accurately predict if someone might have diabetes or is at high risk of diabetes?

In [1]:
ensure_library <- function(pkg) {
  pkg_name <- deparse(substitute(pkg))
  if (!requireNamespace(pkg_name, quietly = TRUE)) {
    install.packages(pkg_name)
  }
  library(pkg_name, character.only = TRUE)
}
library(tidyverse)
data <- read_csv("/data/diabetes/diabetes_binary_5050split_health_indicators_BRFSS2015.csv")


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Rows: 70692 Columns: 22
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (22): Diabetes_binary, HighBP, HighChol, CholCheck, BMI, Smoker, Stroke,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [3]:
p1 <- ggplot(
  data %>% mutate(
    Diabetes = factor(Diabetes_binary),
    `Smoking Status` = c("Non Smoker","Smoker")[Smoker + 1]
  ),
  aes(Diabetes, fill = `Smoking Status`)
) +
  geom_bar(position = "dodge")

p2 <- ggplot(
  data %>% mutate(
    Diabetes = factor(Diabetes_binary),
    `High Blood Pressure` = c("No","Yes")[HighBP + 1]
  ),
  aes(Diabetes, fill = `High Blood Pressure`)
) +
  geom_bar(position = "dodge")

p3 <- ggplot(
  data %>% mutate(
    Diabetes = factor(Diabetes_binary),
    `High Cholesterol` = c("No","Yes")[HighChol + 1]
  ),
  aes(Diabetes, fill = `High Cholesterol`)
) +
  geom_bar(position = "dodge")

p4 <- ggplot(
  data %>% mutate(
    Diabetes = factor(Diabetes_binary),
    `Stroke History` = c("No","Yes")[Stroke + 1]
  ),
  aes(Diabetes, fill = `Stroke History`)
) +
  geom_bar(position = "dodge")

p5 <- ggplot(
  data %>% mutate(
    Diabetes = factor(Diabetes_binary),
    `Heart Disease / Attack` = c("No","Yes")[HeartDiseaseorAttack + 1]
  ),
  aes(Diabetes, fill = `Heart Disease / Attack`)
) +
  geom_bar(position = "dodge")

p6 <- ggplot(
  data %>% mutate(
    Diabetes = factor(Diabetes_binary),
    `Physical Activity` = c("No","Yes")[PhysActivity + 1]
  ),
  aes(Diabetes, fill = `Physical Activity`)
) +
  geom_bar(position = "dodge")

p1
p2
p3
p4
p5
p6

ERROR: Error in UseMethod("mutate"): no applicable method for 'mutate' applied to an object of class "function"


As a first step, try training a standard classification and regression tree on the data. Here is a very bare skeleton of that process.
Note that the following code neglects:

* train/test splits
* imputation of missing data
* potentially variable selection
* model selection

In [19]:
ensure_library(rpart)
f <- formula(sprintf("Diabetes_binary ~ %s", paste0(names(data %>% select(-Diabetes_binary)),collapse=" + ")))
data <- data %>% mutate(Diabetes_binary = factor(Diabetes_binary));

model <- rpart(
  f,
  data = data,
  method = "class"
)
             


In [20]:
f <- formula(sprintf("Diabetes_binary ~ %s", paste0(names(data %>% select(-Diabetes_binary)),collapse=" + ")))
data <- data %>% mutate(Diabetes_binary = factor(Diabetes_binary));

model <- rpart(
  f,
  data = data,
  method = "class"
)
model             


n= 70692 

node), split, n, loss, yval, (yprob)
      * denotes terminal node

 1) root 70692 35346 0 (0.5000000 0.5000000)  
   2) HighBP< 0.5 30860  8742 0 (0.7167207 0.2832793)  
     4) GenHlth< 2.5 17309  2485 0 (0.8564331 0.1435669) *
     5) GenHlth>=2.5 13551  6257 0 (0.5382629 0.4617371)  
      10) Age< 4.5 2104   445 0 (0.7884981 0.2115019) *
      11) Age>=4.5 11447  5635 1 (0.4922687 0.5077313)  
        22) BMI< 27.5 5161  2049 0 (0.6029839 0.3970161) *
        23) BMI>=27.5 6286  2523 1 (0.4013681 0.5986319) *
   3) HighBP>=0.5 39832 13228 1 (0.3320948 0.6679052)  
     6) GenHlth< 2.5 10845  5036 0 (0.5356385 0.4643615)  
      12) BMI< 29.5 6257  2387 0 (0.6185073 0.3814927) *
      13) BMI>=29.5 4588  1939 1 (0.4226242 0.5773758) *
     7) GenHlth>=2.5 28987  7419 1 (0.2559423 0.7440577) *

In [14]:
f

Diabetes_binary ~ HighBP + HighChol + CholCheck + BMI + Smoker + 
    Stroke + HeartDiseaseorAttack + PhysActivity + Fruits + Veggies + 
    HvyAlcoholConsump + AnyHealthcare + NoDocbcCost + GenHlth + 
    MentHlth + PhysHlth + DiffWalk + Sex + Age + Education + 
    Income

In [18]:
model

n= 70692 

node), split, n, loss, yval, (yprob)
      * denotes terminal node

 1) root 70692 35346 0 (0.5000000 0.5000000)  
   2) HighBP< 0.5 30860  8742 0 (0.7167207 0.2832793)  
     4) GenHlth< 2.5 17309  2485 0 (0.8564331 0.1435669) *
     5) GenHlth>=2.5 13551  6257 0 (0.5382629 0.4617371)  
      10) Age< 4.5 2104   445 0 (0.7884981 0.2115019) *
      11) Age>=4.5 11447  5635 1 (0.4922687 0.5077313)  
        22) BMI< 27.5 5161  2049 0 (0.6029839 0.3970161) *
        23) BMI>=27.5 6286  2523 1 (0.4013681 0.5986319) *
   3) HighBP>=0.5 39832 13228 1 (0.3320948 0.6679052)  
     6) GenHlth< 2.5 10845  5036 0 (0.5356385 0.4643615)  
      12) BMI< 29.5 6257  2387 0 (0.6185073 0.3814927) *
      13) BMI>=29.5 4588  1939 1 (0.4226242 0.5773758) *
     7) GenHlth>=2.5 28987  7419 1 (0.2559423 0.7440577) *